In [1]:
import h5py
import numpy as np
import pandas as pd
import os

DATA_DIR = '../../dataset'
BATCH_FILES = [
    '2017-05-12_batchdata_updated_struct_errorcorrect.mat',
    '2017-06-30_batchdata_updated_struct_errorcorrect.mat',
    '2018-04-12_batchdata_updated_struct_errorcorrect.mat',
]
batch_paths = [os.path.join(DATA_DIR, f) for f in BATCH_FILES]

rows = []

for batch_idx, path in enumerate(batch_paths):
    f = h5py.File(path, 'r')
    batch = f['batch']
    n_cells = batch['cycle_life'].shape[0]

    for i in range(n_cells):
        try:
            # target
            cycle_life = f[batch['cycle_life'][i][0]][0][0]

            # summary per cycle
            summary = f[batch['summary'][i][0]]
            Q_discharge = summary['QDischarge'][0][1:]  # skip cycle 0
            IR          = summary['IR'][0][1:]
            Tavg        = summary['Tavg'][0][1:]
            chargetime  = summary['chargetime'][0][1:]
            n_cycles    = len(Q_discharge)

            for c in range(n_cycles):
                rows.append({
                    'cell_id':      f'b{batch_idx+1}_c{i}',
                    'batch':        f'batch_{batch_idx+1}',
                    'cycle_idx':    c + 1,
                    'cycle_life':   cycle_life,   # TARGET
                    'QDischarge':   Q_discharge[c],
                    'IR':           IR[c] if c < len(IR) else np.nan,
                    'Tavg':         Tavg[c] if c < len(Tavg) else np.nan,
                    'chargetime':   chargetime[c] if c < len(chargetime) else np.nan,
                })
        except Exception as e:
            print(f'Skip b{batch_idx+1}_c{i}: {e}')
            continue

    f.close()
    print(f'Batch {batch_idx+1} done: {n_cells} cells')

df = pd.DataFrame(rows)
print(df.shape)
print(df.head(10))

Batch 1 done: 46 cells
Batch 2 done: 48 cells
Batch 3 done: 46 cells
(114598, 8)
  cell_id    batch  cycle_idx  cycle_life  QDischarge        IR       Tavg  \
0   b1_c0  batch_1          1      1190.0    1.070689  0.016742  31.875011   
1   b1_c0  batch_1          2      1190.0    1.071900  0.016724  31.931490   
2   b1_c0  batch_1          3      1190.0    1.072510  0.016681  31.932603   
3   b1_c0  batch_1          4      1190.0    1.073174  0.016662  31.959322   
4   b1_c0  batch_1          5      1190.0    1.073576  0.016623  31.961062   
5   b1_c0  batch_1          6      1190.0    1.073992  0.016600  31.900562   
6   b1_c0  batch_1          7      1190.0    1.074374  0.016577  31.921668   
7   b1_c0  batch_1          8      1190.0    1.074492  0.016588  31.870082   
8   b1_c0  batch_1          9      1190.0    1.074537  0.016572  31.841053   
9   b1_c0  batch_1         10      1190.0    1.074781  0.016565  31.811382   

   chargetime  
0   13.341250  
1   13.425777  
2   13.42516

In [2]:
import os
os.makedirs('../../outputs/v1/dataset', exist_ok=True)
df.to_csv('../../outputs/v1/dataset/battery_summary_dataset.csv', index=False)
print(f'Saved: {df.shape[0]} rows, {df.shape[1]} columns')

Saved: 114598 rows, 8 columns
